# Sync database_files/ from Master_List.csv

`database_files/` holds 8 normalized tables (113 rows each, keyed by `prospect_id` like `ca-001`, `id-001`, ...) that back the dashboard. `Master_List.csv` now has 148 rows (113 original + 36 net-new from the SEA/UZKZ merges).

Approach:
- Match existing `prospect_id` rows to `Master_List.csv` by **name** (`Organisation` or `Alternate Names`), not position — the master file has been reordered since these tables were generated, so positional alignment is wrong.
- Refresh mapped fields on existing rows; add new rows (continuing each region's ID sequence) for the 36 prospects not yet represented.
- Fields with no source column in `Master_List.csv` (`investment_philosophy`, exposure flags, `suggested_conversation_angle`, `next_action`, `next_action_date`, `briefing_pack_status`) are left blank for new rows and untouched for existing rows.
- `assigned_owner` stays `"TBP Advisory"` for all rows (existing and new) — not synced from `Assigned Lead`.
- `priority` in `04_prospect_scores.csv` is overwritten from Master's raw `Priority` column for all rows (existing and new) — confirmed as a deliberate replacement of the old High/Medium scheme.

In [1]:
import shutil
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")
DB_DIR = DATA_DIR / "database_files"
MASTER_PATH = DATA_DIR / "Master_List.csv"


def read_master(path):
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp1252")


master = read_master(MASTER_PATH)

TABLES = {
    "prospects": "01_prospects.csv",
    "profiles": "02_prospect_profiles.csv",
    "sources": "03_prospect_sources.csv",
    "scores": "04_prospect_scores.csv",
    "analysis": "05_prospect_analysis.csv",
    "pipeline": "06_prospect_pipeline.csv",
    "sectors": "07_prospect_sectors.csv",
    "diligence": "08_prospect_diligence.csv",
}

db = {key: pd.read_csv(DB_DIR / fname) for key, fname in TABLES.items()}

print("master:", master.shape)
for key, fname in TABLES.items():
    print(f"{fname}: {db[key].shape}")

master: (148, 32)
01_prospects.csv: (113, 6)
02_prospect_profiles.csv: (113, 13)
03_prospect_sources.csv: (113, 7)
04_prospect_scores.csv: (113, 10)
05_prospect_analysis.csv: (113, 5)
06_prospect_pipeline.csv: (113, 6)
07_prospect_sectors.csv: (332, 2)
08_prospect_diligence.csv: (113, 4)


## Match existing prospects to master rows by name; find net-new prospects

In [2]:
# name (Organisation or Alternate Names) -> master row index
name_to_idx = {}
for i, row in master.iterrows():
    name_to_idx[row["Organisation"]] = i
    if pd.notna(row.get("Alternate Names")):
        for alt in str(row["Alternate Names"]).split(";"):
            alt = alt.strip()
            if alt:
                name_to_idx[alt] = i

prospects = db["prospects"]

# map existing prospect_id -> matched master row index
id_to_master_idx = {}
unmatched_existing = []
for _, r in prospects.iterrows():
    mi = name_to_idx.get(r["prospect_name"])
    if mi is None:
        unmatched_existing.append((r["id"], r["prospect_name"]))
    else:
        id_to_master_idx[r["id"]] = mi

print("existing prospects with no master match:", unmatched_existing)

matched_master_idxs = set(id_to_master_idx.values())
new_master_idxs = [i for i in range(len(master)) if i not in matched_master_idxs]
print(f"\nnet-new prospects to add: {len(new_master_idxs)}")
print(master.loc[new_master_idxs, "Region"].value_counts())

existing prospects with no master match: []

net-new prospects to add: 36
Region
Central Asia    22
Indonesia       14
Name: count, dtype: int64


## Assign new prospect_ids, continuing each region's existing sequence

In [3]:
REGION_PREFIX = {
    "Central Asia": "ca",
    "Indonesia": "id",
    "Malaysia": "my",
    "New York / New Jersey": "us-ny",
    "Singapore": "sg",
    "Sri Lanka": "lk",
}

next_seq = {}
for pid in prospects["id"]:
    prefix, num = pid.rsplit("-", 1)
    next_seq[prefix] = max(next_seq.get(prefix, 0), int(num))

new_id_by_master_idx = {}
for mi in new_master_idxs:
    region = master.loc[mi, "Region"]
    prefix = REGION_PREFIX[region]
    next_seq[prefix] = next_seq.get(prefix, 0) + 1
    new_id_by_master_idx[mi] = f"{prefix}-{next_seq[prefix]:03d}"

print(f"{len(new_id_by_master_idx)} new ids assigned")
list(new_id_by_master_idx.items())[:5]

36 new ids assigned


[(112, 'id-021'),
 (113, 'id-022'),
 (114, 'id-023'),
 (115, 'id-024'),
 (116, 'id-025')]

## 01_prospects.csv — refresh existing + append new

In [4]:
def is_blank(val):
    return pd.isna(val) or (isinstance(val, str) and not val.strip())


prospects = prospects.set_index("id", drop=False)

for pid, mi in id_to_master_idx.items():
    prospects.loc[pid, "prospect_type"] = master.loc[mi, "Prospect Category"]
    prospects.loc[pid, "country"] = master.loc[mi, "Country"]
    prospects.loc[pid, "city"] = master.loc[mi, "HQ / Primary Geography"]
    prospects.loc[pid, "region"] = master.loc[mi, "Region"]

new_prospect_rows = []
for mi, pid in new_id_by_master_idx.items():
    new_prospect_rows.append({
        "id": pid,
        "prospect_name": master.loc[mi, "Organisation"],
        "prospect_type": master.loc[mi, "Prospect Category"],
        "country": master.loc[mi, "Country"],
        "city": master.loc[mi, "HQ / Primary Geography"],
        "region": master.loc[mi, "Region"],
    })

prospects = pd.concat([prospects, pd.DataFrame(new_prospect_rows).set_index("id", drop=False)], ignore_index=True)
db["prospects"] = prospects

print(db["prospects"].shape)
db["prospects"].tail(5)

(149, 6)


,id,prospect_name,prospect_type,country,city,region
144,ca-036,Samruk-Kazyna,Sovereign wealth / strategic holding,Kazakhstan,Astana,Central Asia
145,ca-037,Baiterek Holding,Development holding,Kazakhstan,Astana,Central Asia
146,ca-038,Kazakhstan Temir Zholy (KTZ),National rail and logistics operator,Kazakhstan,Astana,Central Asia
147,ca-039,Kazakh Invest,Investment promotion agency,Kazakhstan,Astana,Central Asia
148,ca-040,Almaty Industrial Zone / industrial parks ecos...,Industrial zone / corridor node,Kazakhstan,Almaty,Central Asia


## Generic sync for single-row-per-prospect tables (02, 03, 04, 05, 06)

For each table, only columns listed in `mapping` are touched. Unmapped columns (e.g. `investment_philosophy`, `assigned_owner`, `next_action_date`) are left as-is on existing rows and blank on new rows.

In [5]:
def sync_simple_table(table_key, mapping):
    table = db[table_key].set_index("prospect_id", drop=False)
    all_cols = list(db[table_key].columns)

    for pid, mi in id_to_master_idx.items():
        for db_col, master_col in mapping.items():
            table.loc[pid, db_col] = master.loc[mi, master_col]

    new_rows = []
    for mi, pid in new_id_by_master_idx.items():
        row = {col: None for col in all_cols}
        row["prospect_id"] = pid
        for db_col, master_col in mapping.items():
            row[db_col] = master.loc[mi, master_col]
        new_rows.append(row)

    table = pd.concat(
        [table, pd.DataFrame(new_rows, columns=all_cols).set_index("prospect_id", drop=False)],
        ignore_index=True,
    )
    db[table_key] = table[all_cols]
    print(table_key, "->", db[table_key].shape)


sync_simple_table("profiles", {
    "family_or_group_background": "Family / Founder / Strategic Nature",
})

sync_simple_table("sources", {
    "public_source_url": "Public Source URLs",
    "source_quality": "Contact Email Status",
    "email": "Email address",
    "address": "Address / Office Location",
})

sync_simple_table("scores", {
    "suitability_score": "Total Score",
    "classification": "Classification",
    "priority": "Priority",
    "family_office_fit": "Family Office Fit (20)",
    "permanent_capital_orientation": "Permanent Capital (20)",
    "sector_alignment": "Sector Alignment (20)",
    "governance_institutional_mindset": "Governance Mindset (15)",
    "strategic_adjacency_tbp": "Strategic Adjacency (15)",
    "engagement_readiness": "Engagement Readiness (10)",
})

sync_simple_table("analysis", {
    "tbp_relevance_summary": "TBP / Regional Corridor Relevance",
    "best_tbp_entry_point": "Possible TBP Entry Point",
    "recommended_contact_route": "Recommended Contact Route",
})

sync_simple_table("pipeline", {
    "pipeline_stage": "Pipeline Stage",
})

# assigned_owner stays "TBP Advisory" for everyone, including new rows (explicit decision, not sourced from Assigned Lead)
new_pipeline_ids = list(new_id_by_master_idx.values())
db["pipeline"].loc[db["pipeline"]["prospect_id"].isin(new_pipeline_ids), "assigned_owner"] = "TBP Advisory"

profiles -> (149, 13)
sources -> (149, 7)


scores -> (149, 10)


analysis -> (149, 5)
pipeline -> (149, 6)


## 07_prospect_sectors.csv — regenerate from Known Sector Themes

Rebuilt in full for every prospect that has a master match (existing + new), since it's cheap and guarantees correctness after the one merge-affected row (SMDV) changed its sector list.

In [6]:
# combine existing (matched) + new prospect_id -> master row index
all_id_to_master_idx = dict(id_to_master_idx)
for mi, pid in new_id_by_master_idx.items():
    all_id_to_master_idx[pid] = mi

sector_rows = []
for pid, mi in all_id_to_master_idx.items():
    themes = master.loc[mi, "Known Sector Themes"]
    if pd.isna(themes):
        continue
    for sector in str(themes).split(";"):
        sector = sector.strip()
        if sector:
            sector_rows.append({"prospect_id": pid, "sector": sector})

# keep sector rows for any prospect_id that had no master match (untouched)
unmatched_ids = {pid for pid, _ in unmatched_existing}
preserved = db["sectors"][db["sectors"]["prospect_id"].isin(unmatched_ids)]

db["sectors"] = pd.concat([pd.DataFrame(sector_rows), preserved], ignore_index=True)
print("sectors ->", db["sectors"].shape)

sectors -> (369, 2)


## 08_prospect_diligence.csv — refresh content, add new rows

In [7]:
diligence = db["diligence"].set_index("prospect_id", drop=False)

for pid, mi in id_to_master_idx.items():
    diligence.loc[pid, "content"] = master.loc[mi, "Notes / Diligence Flags"]

new_diligence_rows = []
for mi, pid in new_id_by_master_idx.items():
    new_diligence_rows.append({
        "prospect_id": pid,
        "type": "flag",
        "content": master.loc[mi, "Notes / Diligence Flags"],
        "sort_order": 1,
    })

diligence = pd.concat(
    [diligence, pd.DataFrame(new_diligence_rows).set_index("prospect_id", drop=False)],
    ignore_index=True,
)
db["diligence"] = diligence[["prospect_id", "type", "content", "sort_order"]]
print("diligence ->", db["diligence"].shape)

diligence -> (149, 4)


## Back up and write all 8 tables

In [8]:
for key, fname in TABLES.items():
    path = DB_DIR / fname
    backup_path = path.with_name(path.stem + ".backup.csv")
    if not backup_path.exists():
        shutil.copy(path, backup_path)
        print("Backed up ->", backup_path)
    else:
        print("Backup already exists, skipping ->", backup_path)

for key, fname in TABLES.items():
    db[key].to_csv(DB_DIR / fname, index=False)
    print("Wrote", fname, "rows:", len(db[key]))

Backed up -> data\database_files\01_prospects.backup.csv
Backed up -> data\database_files\02_prospect_profiles.backup.csv
Backed up -> data\database_files\03_prospect_sources.backup.csv
Backed up -> data\database_files\04_prospect_scores.backup.csv
Backed up -> data\database_files\05_prospect_analysis.backup.csv
Backed up -> data\database_files\06_prospect_pipeline.backup.csv
Backed up -> data\database_files\07_prospect_sectors.backup.csv
Backed up -> data\database_files\08_prospect_diligence.backup.csv
Wrote 01_prospects.csv rows: 149
Wrote 02_prospect_profiles.csv rows: 149
Wrote 03_prospect_sources.csv rows: 149
Wrote 04_prospect_scores.csv rows: 149
Wrote 05_prospect_analysis.csv rows: 149
Wrote 06_prospect_pipeline.csv rows: 149
Wrote 07_prospect_sectors.csv rows: 369
Wrote 08_prospect_diligence.csv rows: 149


## Verify: reload from disk and spot-check

In [9]:
reloaded = {key: pd.read_csv(DB_DIR / fname) for key, fname in TABLES.items()}
for key, fname in TABLES.items():
    print(fname, reloaded[key].shape)

print()
new_ids_sample = list(new_id_by_master_idx.values())[:3]
print("sample new prospects:")
print(reloaded["prospects"][reloaded["prospects"]["id"].isin(new_ids_sample)])
print()
print("sample new scores (priority should show master's raw tier labels):")
print(reloaded["scores"][reloaded["scores"]["prospect_id"].isin(new_ids_sample)][["prospect_id", "suitability_score", "priority"]])
print()
print("sample new pipeline (assigned_owner should be TBP Advisory):")
print(reloaded["pipeline"][reloaded["pipeline"]["prospect_id"].isin(new_ids_sample)])

01_prospects.csv (149, 6)
02_prospect_profiles.csv (149, 13)
03_prospect_sources.csv (149, 7)
04_prospect_scores.csv (149, 10)
05_prospect_analysis.csv (149, 5)
06_prospect_pipeline.csv (149, 6)
07_prospect_sectors.csv (369, 2)
08_prospect_diligence.csv (149, 4)

sample new prospects:
         id                               prospect_name  \
113  id-021                   PT Samudera Indonesia Tbk   
114  id-022  PT Centratama Telekomunikasi Indonesia Tbk   
115  id-023                    PT Adi Sarana Armada Tbk   

                    prospect_type    country     city     region  
113  Listed Corporate (IDX: SMDR)  Indonesia  Jakarta  Indonesia  
114  Listed Corporate (IDX: CENT)  Indonesia  Jakarta  Indonesia  
115  Listed Corporate (IDX: ASSA)  Indonesia  Jakarta  Indonesia  

sample new scores (priority should show master's raw tier labels):
    prospect_id  suitability_score     priority
113      id-021                 88         High
114      id-022                 77  Medium-Hi